# Fixedpoint：把规则写成递归，让引擎去推可达性

## 这一章要认识的新引擎

Z3 里有个叫 `Fixedpoint` 的引擎，专门处理**递归的规则**。“从 A 能走到 B，从 B 能走到 C，那从 A 也能走到 C”，这种推理你不用自己写循环，原样写成规则交给它就行。

- 关系用规则（`rule`）和事实（`fact`）来描述
- 想知道某件事成不成立，`query()` 问一声
- 能到达的时候，具体路径从哪里掏出来

它默认跑的是**自底向上的 Datalog**。Datalog 跟关系数据库的查询（SQL 里的递归查询）是同一路子：你把事实摆上去，它反复套用规则，不断长出新的东西来。

In [1]:
from z3 import *

## 例 1：图的可达性

看下面这张无向图，问题是：从顶点 1 能不能走到顶点 6？

```{mermaid}
flowchart LR
1 --- 2
1 --- 5
5 --- 2
4 --- 5
2 --- 3
4 --- 3
4 --- 6
```

这里要用到三个谓词。

- `edge(a, b)` … `a` 和 `b` 之间有一条边（当事实给进去）
- `connect(a, b)` … 从 `a` 能直接挪到 `b`（有边就两个方向都成立）
- `path(a, b)` … 从 `a` 能到达 `b`（这个靠递归定下来）

In [2]:
fp = Fixedpoint()
fp.set(engine='datalog')          # 使用 Datalog 引擎

s = BitVecSort(8)                  # 顶点编号用 8 位表示

connect = Function('connect', s, s, BoolSort())
edge = Function('edge', s, s, BoolSort())
path = Function('path', s, s, BoolSort())

a, b, c = Consts('a b c', s)

fp.register_relation(path, edge, connect)
fp.declare_var(a, b, c)

# 规则：有边就能直接移动（无向，所以两个方向都成立）
fp.rule(connect(a, b), edge(a, b))
fp.rule(connect(a, b), edge(b, a))

# 规则：能直接移动，就能到达
fp.rule(path(a, b), connect(a, b))
# 规则：a→b 能移动，且 b→c 能到达，则 a→c 能到达
fp.rule(path(a, c), [connect(a, b), path(b, c)])

# 事实：图的边
edges = [(1, 2), (1, 5), (5, 2), (4, 5), (2, 3), (4, 3), (4, 6)]
for p, q in edges:
    fp.fact(edge(p, q))

### 代码说明

**1. 先挑引擎**

```python
fp = Fixedpoint()
fp.set(engine='datalog')
```

`engine='datalog'` 这一句，就是把 Datalog 引擎点出来。

**2. 定顶点的类型**

```python
s = BitVecSort(8)
```

顶点编号用 8 位的位向量装。Datalog 只认有限的类型，所以这里得用 `BitVecSort` 这样的有限类型，不能拿 `IntSort` 来顶。

**3. 定义谓词（关系）**

```python
connect = Function('connect', s, s, BoolSort())
```

`Function(名称, 参数的类型..., 返回值的类型)` 用来造谓词。connect 是“吃两个顶点、吐一个真假”的关系，所以返回值写 `BoolSort()`。

**4. 声明变量、注册谓词**

```python
fp.register_relation(path, edge, connect)
fp.declare_var(a, b, c)
```

谓词得先让引擎认下来，这一步是 `register_relation()`；规则里要用到的变量，则由 `declare_var()` 声明。

**5. 规则与事实**

```python
fp.rule(path(a, c), [connect(a, b), path(b, c)])
```

`rule(头, 本体)` 读作“本体全都成立，头就成立”。上面这条说的是：从 `a` 移到 `b`，再从 `b` 到 `c`，那么 `a` 到 `c` 也通。本体里出现了 `path` 自己，递归就是这么来的。

至于 `fp.fact(edge(1, 2))`，它把“这里有条边”当事实登记进去。

### 查询

In [3]:
print(fp.query(path(1, 6)))     # 能否到达
print(fp.query(path(1, 7)))     # 不存在的顶点

sat
unsat


顶点 1 到得了 6，回的是 `sat`；顶点 7 到不了，回的是 `unsat`。这种“成不成立”的问题，正是 Fixedpoint 的本职。

## 把路径重建出来

能到达是知道了，**可到底该怎么走**，还不知道。

Z3 的 Datalog 本来有个输出**说明（explanations）**的功能，能讲清某条事实为什么成立（`datalog.generate_explanations`）。可惜在最近的 Z3 里（这个环境是 5.1.0）它不太稳：有时报 `explanations are not supported with undefined predicates`，有时把答案变成 `unsat`，闹得不好还会让 Z3 整个挂掉。

所以这一章的路子是自己把路径做出来：可达与否交给 Fixedpoint，具体走法把边列表做一遍 BFS（广度优先搜索）就有。

In [4]:
from collections import deque


def bfs_route(edges, start, goal, undirected=True):
    """从 (from, to) 的列表出发，用 BFS 构造最短路径"""
    neighbors = {}
    for p, q in edges:
        neighbors.setdefault(p, []).append(q)
        if undirected:
            neighbors.setdefault(q, []).append(p)

    prev = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == goal:
            break
        for nxt in neighbors.get(cur, []):
            if nxt not in prev:
                prev[nxt] = cur
                queue.append(nxt)

    if goal not in prev:
        return None

    route = [goal]
    while prev[route[-1]] is not None:
        route.append(prev[route[-1]])
    return route[::-1]


print(bfs_route(edges, 1, 6))

[1, 5, 4, 6]


算出来的路径是 **1 → 5 → 4 → 6**。`edge(4, 5)` 是无向边，5→4 一样走得通。

## 例 2：三个水桶

手上有 8 升、5 升、3 升三个桶。起初只有 8 升那个是满的，另外两个空着。要让**两个桶各装 4 升**，水该怎么倒？

这其实就是一张图上的可达性问题：状态当顶点，倒水当边。先用 Python 把所有可能的状态、以及状态之间的转移全列出来。

- 不管怎么倒，水量合计永远是 8 升
- 能做的只有一件事：从一个桶往另一个桶里倒
- 倒完只有两种结局：**目标桶满了**，或者**来源桶空了**

In [5]:
from itertools import product

sizes = (8, 5, 3)

# 可能的状态（合计 8 升）
states = [(a, b, c)
          for a, b, c in product(*[range(size + 1) for size in sizes])
          if a + b + c == 8]

# 一次倒水能到达的状态对
transitions = []
for s1, s2 in product(states, states):
    changed = [(size, old, new) for size, old, new in zip(sizes, s1, s2) if old != new]
    if len(changed) != 2:
        continue        # 变化的桶正好 2 个（倒出来源和倒进目标）
    if any(new == size or new == 0 for size, old, new in changed):
        transitions.append((s1, s2))    # 目标桶满 or 来源桶空

print(len(states), "种状态，", len(transitions), "种倒法")

24 种状态， 106 种倒法


接下来把这些转移当成 `pour` 谓词的**事实**注册进去，再用 `plan` 谓词推一件事：从初始状态出发，到得了目标状态吗？

- `pour(a, b, c, a2, b2, c2)` … 从状态 `(a, b, c)` 一步就能移到 `(a2, b2, c2)`
- `plan(a, b, c, a2, b2, c2)` … 从状态 `(a, b, c)` 走若干步能移到 `(a2, b2, c2)`

In [6]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(pour, plan)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

# 一步能移，则 plan 也成立
fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
# 移一步之后还有 plan，则 plan 成立（递归）
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

for s1, s2 in transitions:
    fp.fact(pour(*s1, *s2))

start, goal = (8, 0, 0), (4, 4, 0)

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 0 ns
Wall time: 1.78 ms


回了 `sat`，说明从 8,0,0 到 4,4,0 之间有路可走。

路怎么找？拿 `transitions`（那张边列表）做 BFS 就行。例 1 写的 `bfs_route()` 直接搬过来用，只是这里是有向图，参数要写 `undirected=False`。

In [7]:
route = bfs_route(transitions, start, goal, undirected=False)

for s1, s2 in zip(route, route[1:]):
    print(s1, "->", s2)

(8, 0, 0) -> (3, 5, 0)
(3, 5, 0) -> (3, 2, 3)
(3, 2, 3) -> (6, 2, 0)
(6, 2, 0) -> (6, 0, 2)
(6, 0, 2) -> (1, 5, 2)
(1, 5, 2) -> (1, 4, 3)
(1, 4, 3) -> (4, 4, 0)


七次倒水，4,4,0 就到了。人来做这种题得反复试错，可**只要状态和转移写得出来**，剩下的是机械活。

顺手还能把注册进去的规则和事实翻出来看看。

In [8]:
for rule in fp.get_rules()[:5]:
    print(rule)
print("...")

ForAll([A, B, C, D, E, F],
       Implies(pour(A, B, C, D, E, F),
               plan(A, B, C, D, E, F)))
ForAll([A, B, C, D, E, F, G, H, I],
       Implies(And(plan(D, E, F, G, H, I),
                   pour(A, B, C, D, E, F)),
               plan(A, B, C, G, H, I)))
pour(0, 5, 3, 3, 5, 0)
pour(0, 5, 3, 5, 0, 3)
pour(1, 4, 3, 0, 5, 3)
...


## 例 3：让 `pour` 变成规则

例 2 是所有转移都用 Python 算好，再把 `pour` 当事实塞进去。这回反过来，**`pour` 用规则写**：可能的状态 `state(a, b, c)` 作为事实给出，倒水的条件写成规则。

条件是“变化的桶有两个”，而且其中一个要么满了，要么空了。

In [9]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
state = Function('state', s, s, s, BoolSort())
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(state, pour, plan)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

# 倒水规则：变化的桶有 2 个，其中一个变满或变空
fp.rule(
    pour(a, b, c, a1, b1, c1),
    [
        state(a, b, c),
        state(a1, b1, c1),
        Or(
            And(a == a1, b != b1, Or(b1 == 0, b1 == 5, c1 == 0, c1 == 3)),
            And(b == b1, a != a1, Or(a1 == 0, a1 == 8, c1 == 0, c1 == 3)),
            And(c == c1, a != a1, Or(a1 == 0, a1 == 8, b1 == 0, b1 == 5)),
        ),
    ],
)

# 把可能的状态（合计 8 升）注册为事实
for p in states:
    fp.fact(state(*p))

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 62.5 ms
Wall time: 47.1 ms


跟例 2 一样是 `sat`。`pour` 换成规则写，只要状态仍然当事实给，结论一个样。这个环境里花了 0.04 秒左右（例 2 是 1 毫秒上下）。

读 `pour` 那条规则，眼睛要盯住本体里的 `state(a, b, c)` 和 `state(a1, b1, c1)`：**两个都得在**。倒完之后的那个状态，同样得是可能的状态。

## 例 4：连 `state` 也写成规则

最后一步，把 `state` 也交给规则。每个桶各自能装多少，用 `bucket_a`、`bucket_b`、`bucket_c` 的事实给出，`state` 则由它们的组合（合计 8 升）推出来。

In [10]:
fp = Fixedpoint()
fp.set(engine='datalog')

s = BitVecSort(5)
bucket_a = Function('bucket_a', s, BoolSort())
bucket_b = Function('bucket_b', s, BoolSort())
bucket_c = Function('bucket_c', s, BoolSort())
state = Function('state', s, s, s, BoolSort())
pour = Function('pour', s, s, s, s, s, s, BoolSort())
plan = Function('plan', s, s, s, s, s, s, BoolSort())
a, b, c, a1, b1, c1, a2, b2, c2 = Consts(['a', 'b', 'c', 'a1', 'b1', 'c1', 'a2', 'b2', 'c2'], s)

fp.register_relation(state, pour, plan, bucket_a, bucket_b, bucket_c)
fp.declare_var(a, b, c, a1, b1, c1, a2, b2, c2)

fp.rule(plan(a, b, c, a2, b2, c2), pour(a, b, c, a2, b2, c2))
fp.rule(plan(a, b, c, a2, b2, c2), [pour(a, b, c, a1, b1, c1), plan(a1, b1, c1, a2, b2, c2)])

fp.rule(
    pour(a, b, c, a1, b1, c1),
    [
        state(a, b, c),
        state(a1, b1, c1),
        Or(
            And(a == a1, b != b1, Or(b1 == 0, b1 == 5, c1 == 0, c1 == 3)),
            And(b == b1, a != a1, Or(a1 == 0, a1 == 8, c1 == 0, c1 == 3)),
            And(c == c1, a != a1, Or(a1 == 0, a1 == 8, b1 == 0, b1 == 5)),
        ),
    ],
)

# state 是“各桶水量的组合”，且合计为 8 升
fp.rule(state(a, b, c), [bucket_a(a), bucket_b(b), bucket_c(c), a + b + c == 8])

for i in range(sizes[0] + 1):
    fp.fact(bucket_a(i), str(i))
for i in range(sizes[1] + 1):
    fp.fact(bucket_b(i), str(i))
for i in range(sizes[2] + 1):
    fp.fact(bucket_c(i), str(i))

%time print(fp.query(plan(*start, *goal)))

sat
CPU times: total: 844 ms
Wall time: 832 ms


结果还是 `sat`，但**慢得多了**。这个环境里 0.6 秒，例 3 只要 0.04 秒，差出 15 倍左右。

原因不难想：`state` 成了规则之后，每应用一次 `pour` 的规则，`state` 的组合就要重新考虑一遍，搜索量自然涨上去。“能写成规则”和“跑得快”是两件事，这个例子正好说明：**已经握在手里的关系，当事实给进去更划算**。

## 收个尾

| 用到的东西 | 干什么用 |
|---|---|
| `Fixedpoint()` | 创建引擎 |
| `set(engine='datalog')` | 选择 Datalog 引擎 |
| `Function(名称, 类型..., BoolSort())` | 创建谓词 |
| `register_relation(...)` | 把谓词注册到引擎 |
| `declare_var(...)` | 声明规则里用的变量 |
| `rule(头, 本体)` | 添加“本体成立则头成立”的规则 |
| `fact(...)` | 添加事实 |
| `query(...)` | 询问是否成立（`sat` / `unsat`） |
| `get_rules()` | 确认注册的规则和事实 |

- 可达性拆成两条规则就够：“一步能挪”加上“递归”
- Fixedpoint 只回答“到不到得了”。具体路径，拿边列表做 BFS 自己造
- 靠说明（`generate_explanations`）取路径这招，在最近的 Z3 里不稳，别太指望
- 状态和转移都清楚的时候，当事实给更快。全写成规则灵活是灵活，就是慢